In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

import pandas as pd
import pytesseract
from PIL import Image
import io
import re
import time


columns = ["Rank", "Game", "Total Prize Money", "Total Players Competed", "Total Number of Tournaments"]

def clean_ocr_text(text):
    if not text:
        return ""
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_image(image):
    image = image.convert("L")  # grayscale
    image = image.point(lambda x: 0 if x < 150 else 255, "1")  # threshold
    return image

def get_ocr_from_element(element):
    #OCR from img if possible, otherwise OCR whole element
    try:
        img = element.find_element(By.TAG_NAME, "img")
        png = img.screenshot_as_png
        image = Image.open(io.BytesIO(png))
        image = preprocess_image(image)
        ocr_text = pytesseract.image_to_string(image, config="--psm 7")
        return clean_ocr_text(ocr_text)
    except:
        try:
            png = element.screenshot_as_png
            image = Image.open(io.BytesIO(png))
            image = preprocess_image(image)
            ocr_text = pytesseract.image_to_string(image, config="--psm 6")
            return clean_ocr_text(ocr_text)
        except:
            return ""

def parse_top5_from_page(driver):
    
    # The first 5 games are not in the table. We extract them from the visible page text.
    page_text = driver.find_element(By.TAG_NAME, "body").text

    # Regex pattern for top 5 featured blocks
    pattern = re.compile(
        r"#(\d+)\s+([^\n]+)\s+\$([\d,]+\.\d{2})\s+(\d[\d,]*) Players\s+(\d[\d,]*) Tournaments",
        re.MULTILINE
    )

    matches = pattern.findall(page_text)

    top5_data = []
    for match in matches[:5]:
        rank, game, prize, players, tournaments = match

        top5_data.append({
            "Rank": rank,
            "Game": game.strip(),
            "Total Prize Money": f"${prize}",
            "Total Players Competed": f"{players}",
            "Total Number of Tournaments": f"{tournaments}",
        })

    return top5_data

def get_game_details_from_row(row):
    cells = row.find_elements(By.TAG_NAME, "td")
    texts = [cell.text.strip() for cell in cells]

    def clean_stat(text):
        return (
            text.replace("Players", "")
                .replace("Tournaments", "")
                .replace("\n", " ")
                .strip()
        )

    if len(texts) < 5:
        return None

    ocr_text = get_ocr_from_element(row)

    return {
        "Rank": texts[0],
        "Game": texts[1],
        "Total Prize Money": texts[2],
        "Total Players Competed": clean_stat(texts[3]),
        "Total Number of Tournaments": clean_stat(texts[4]),
    }

def main():
    game_data = []

    options = Options()
    options.add_argument("--headless=new")                          # The browser window keeps flickering without this 
    options.add_argument("--window-size=1800,4000")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")     

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
       # Get the URL
        url = "https://www.esportsearnings.com/games"
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(3)

        # Top 5 games
        top5_games = parse_top5_from_page(driver)
        game_data.extend(top5_games)

        print(f"Captured top 5 featured games: {len(top5_games)}")

        
        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

        for row in rows:
            game = get_game_details_from_row(row)
            if game:
                game_data.append(game)

            if len(game_data) >= 100:
                break

    finally:
        driver.quit()

    
    df = pd.DataFrame(game_data, columns=columns)

    # Convert rank to numeric and sort
    df["Rank"] = pd.to_numeric(df["Rank"], errors="coerce").fillna(0).astype(int)
    df = df.sort_values("Rank").reset_index(drop=True)

    # Keep only top 100
    df = df.head(100)

    df.to_csv("top_games_awarding_prize_money.csv", index=False, encoding="utf-8-sig")

    print(df.head(10))
    print(f"Saved {len(df)} games to top_games_awarding_prize_money.csv")

if __name__ == "__main__":
    main()

Captured top 5 featured games: 5
   Rank                                  Game Total Prize Money  \
0     1                                Dota 2   $381,278,250.96   
1     2                              Fortnite   $203,089,308.90   
2     3      Counter-Strike: Global Offensive   $162,769,575.27   
3     4                     League of Legends   $121,953,068.43   
4     5                        Arena of Valor   $111,511,257.36   
5     6  PLAYERUNKNOWN'S BATTLEGROUNDS Mobile    $89,785,122.22   
6     7         PLAYERUNKNOWN’S BATTLEGROUNDS    $67,037,707.91   
7     8                         Rocket League    $54,078,530.56   
8     9                     Rainbow Six Siege    $49,885,252.94   
9    10                          StarCraft II    $43,451,866.17   

  Total Players Competed Total Number of Tournaments  
0                  5069                         1986  
1                 10984                         2586  
2                 16582                         7108  
3        